# Lesson 3.1 — Template Functions & Pipelines

**Phase 3: Advanced Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Use Go template pipelines (`|`) to chain functions
- Apply common Helm/Sprig functions: `default`, `quote`, `upper`, `lower`, `trim`, `indent`, `nindent`, `toYaml`, `required`
- Understand why certain functions are critical for production charts

---

## Pipelines

In Helm templates, the `|` pipe sends the result of one expression as the **last argument** of the next function:

```
{{ .Values.name | default "world" | upper | quote }}
```

This reads as: take `.Values.name` → if empty use `"world"` → convert to uppercase → wrap in quotes.

> **Reference:** [Template Functions and Pipelines](https://helm.sh/docs/chart_template_guide/functions_and_pipelines/) and [Sprig Library](https://masterminds.github.io/sprig/)

---

## Setup

In [ ]:
rm -rf /tmp/funcdemo && mkdir -p /tmp/funcdemo/templates

cat > /tmp/funcdemo/Chart.yaml << 'EOF'
apiVersion: v2
name: funcdemo
version: 0.1.0
appVersion: "1.0"
EOF

cat > /tmp/funcdemo/values.yaml << 'EOF'
appName: ""
environment: ""
replicas: 2
config:
  logLevel: info
  maxConnections: 100
labels:
  team: platform
  tier: backend
EOF

echo "Setup complete"

---

## `default` — Provide Fallback Values

In [ ]:
cat > /tmp/funcdemo/templates/defaults-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: defaults-demo
data:
  # If .Values.appName is empty, use "myapp"
  appName: {{ .Values.appName | default "myapp" | quote }}

  # If .Values.environment is empty, use "development"
  environment: {{ .Values.environment | default "development" | quote }}

  # Numbers don't need quoting but can use default
  replicas: {{ .Values.replicas | default 1 }}
EOF

echo "=== With defaults (values are empty strings) ==="
helm template my-release /tmp/funcdemo --show-only templates/defaults-demo.yaml

echo ""
echo "=== With --set overrides ==="
helm template my-release /tmp/funcdemo \
  --show-only templates/defaults-demo.yaml \
  --set appName=orders-service \
  --set environment=production

---

## `required` — Fail Fast on Missing Values

Use `required` when a value has no sensible default and must be provided by the user:

In [ ]:
cat > /tmp/funcdemo/templates/required-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: required-demo
data:
  # This will fail with a clear error message if .Values.databaseUrl is not set
  databaseUrl: {{ required "databaseUrl is required — set it with --set databaseUrl=<url>" .Values.databaseUrl | quote }}
EOF

echo "=== Should fail with a helpful error message ==="
helm template my-release /tmp/funcdemo \
  --show-only templates/required-demo.yaml 2>&1 || true

echo ""
echo "=== Should succeed when value is provided ==="
helm template my-release /tmp/funcdemo \
  --show-only templates/required-demo.yaml \
  --set databaseUrl=postgres://db:5432/mydb

---

## String Functions

In [ ]:
cat > /tmp/funcdemo/templates/strings-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: strings-demo
data:
  # upper / lower
  upperEnv: {{ .Values.environment | default "dev" | upper | quote }}
  lowerChart: {{ .Chart.Name | lower | quote }}

  # trim whitespace
  trimmed: {{ "  hello world  " | trim | quote }}

  # replace
  dashedName: {{ .Chart.Name | replace "-" "_" | quote }}

  # trunc + trimSuffix — common pattern for Kubernetes name length limits (63 chars)
  safeName: {{ printf "%s-%s" .Release.Name .Chart.Name | trunc 63 | trimSuffix "-" | quote }}

  # contains check
  isProduction: {{ .Values.environment | default "dev" | contains "prod" | quote }}
EOF

helm template my-release /tmp/funcdemo \
  --show-only templates/strings-demo.yaml \
  --set environment=production

---

## `toYaml` + `indent` / `nindent` — Embedding YAML Blocks

In [ ]:
cat > /tmp/funcdemo/templates/toyaml-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: toyaml-demo
  # toYaml converts a map/list value to a YAML string.
  # nindent adds a leading newline + indentation — perfect for embedding blocks.
  labels:
    {{- toYaml .Values.labels | nindent 4 }}
data:
  # The 'config' map is written as a multi-line YAML block
  config: |
    {{- toYaml .Values.config | nindent 4 }}
EOF

helm template my-release /tmp/funcdemo \
  --show-only templates/toyaml-demo.yaml

> **Key pattern:** `{{- toYaml .Values.someMap | nindent <n> }}` is the standard way to safely embed arbitrary YAML maps or lists into templates. You will see this pattern in almost every real-world chart.

---

## `printf` — String Formatting

In [ ]:
cat > /tmp/funcdemo/templates/printf-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: printf-demo
data:
  # printf works like Go's fmt.Sprintf
  imageFull: {{ printf "%s:%s" .Values.image.repository (.Values.image.tag | default .Chart.AppVersion) | quote }}
  greeting: {{ printf "Hello from %s v%s!" .Chart.Name .Chart.AppVersion | quote }}
EOF

cat >> /tmp/funcdemo/values.yaml << 'EOF'
image:
  repository: myregistry/myapp
  tag: ""
EOF

helm template my-release /tmp/funcdemo \
  --show-only templates/printf-demo.yaml

---

## Quick Reference: Most-Used Helm Functions

| Function | Example | Result |
|---|---|---|
| `default` | `"" \| default "x"` | `"x"` |
| `required` | `required "msg" .Values.x` | error if empty |
| `quote` | `"hello" \| quote` | `"\"hello\""` |
| `upper` | `"hello" \| upper` | `"HELLO"` |
| `lower` | `"HELLO" \| lower` | `"hello"` |
| `trim` | `"  x  " \| trim` | `"x"` |
| `trunc 63` | `"long-name" \| trunc 63` | max 63 chars |
| `replace` | `"a-b" \| replace "-" "_"` | `"a_b"` |
| `contains` | `"prod" \| contains "p"` | `true` |
| `printf` | `printf "%s:%s" a b` | `"a:b"` |
| `toYaml` | `toYaml .Values.map` | YAML string |
| `nindent n` | `str \| nindent 4` | newline + 4 spaces indent |
| `indent n` | `str \| indent 4` | 4 spaces indent (no leading newline) |

---

## Exercise

In [ ]:
# TASK 1: Create a template that builds a safe Kubernetes name:
#   - Combine .Release.Name and .Chart.Name with a dash
#   - Truncate to 63 characters
#   - Trim any trailing dash
#   - Lowercase the result
# TODO:


# TASK 2: Create a template where .Values.dbPassword is 'required'
#   and verify that helm template fails with a useful error message
#   when the value is not provided.
# TODO:


# TASK 3: Add an 'annotations' map to values.yaml (with 2-3 entries)
#   and use toYaml + nindent to embed it into a ConfigMap's metadata.annotations.
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- Pipelines chain functions: `value | func1 | func2 | func3`.
- `default` provides fallbacks; `required` fails fast with a helpful message.
- `toYaml | nindent n` is the standard pattern for embedding maps and lists from values.
- String functions (`upper`, `lower`, `trim`, `trunc`, `replace`) and `printf` handle all naming patterns.

---

## Next Lesson

➡️ [Lesson 3.2 — Control Flow: if, with, range](lesson-3.2-control-flow.ipynb)